In [ ]:
pip install pyshark pandas matplotlib nest_asyncio

ERROR: Could not find a version that satisfies the requirement tshark (from versions: none)
ERROR: No matching distribution found for tshark
Note: you may need to restart the kernel to use updated packages.


In [6]:
import nest_asyncio
nest_asyncio.apply()

import pyshark

pcap_path = './dnssec.pcap'
cap = pyshark.FileCapture(pcap_path)


In [9]:
for packet in cap:
    print(packet)

Packet (Length: 83)
Layer ETH
:	Destination: 00:90:0b:12:91:c1
	Address: 00:90:0b:12:91:c1
	.... ..0. .... .... .... .... = LG bit: Globally unique address (factory default)
	.... ...0 .... .... .... .... = IG bit: Individual address (unicast)
	Source: 00:1c:c0:93:33:fb
	.... ..0. .... .... .... .... = LG bit: Globally unique address (factory default)
	.... ...0 .... .... .... .... = IG bit: Individual address (unicast)
	Type: IPv4 (0x0800)
	Address: 00:1c:c0:93:33:fb
Layer IP
:	0100 .... = Version: 4
	.... 0101 = Header Length: 20 bytes (5)
	Differentiated Services Field: 0x00 (DSCP: CS0, ECN: Not-ECT)
	0000 00.. = Differentiated Services Codepoint: Default (0)
	.... ..00 = Explicit Congestion Notification: Not ECN-Capable Transport (0)
	Total Length: 69
	Identification: 0xce58 (52824)
	000. .... = Flags: 0x0
	0... .... = Reserved bit: Not set
	.0.. .... = Don't fragment: Not set
	..0. .... = More fragments: Not set
	...0 0000 0000 0000 = Fragment Offset: 0
	Time to Live: 64
	Protocol

In [11]:
for packet in cap:
    if 'DNS' in packet:
        print(packet.dns.qry_name)  # query name


www.ietf.org
www.ietf.org
ietf.org
ietf.org
ietf.org
ietf.org
org
org
org
org
<Root>
<Root>
<Root>
<Root>


In [12]:
for packet in cap:
    if hasattr(packet.dns, 'resp_type'):
        print(packet.dns.resp_type)  # record type, e.g. RRSIG, DNSKEY

41
1
41
48
41
43
41
48
41
43
41
48
41
47


In [13]:
import nest_asyncio
nest_asyncio.apply()

import pyshark

pcap_path = './dnssec.pcap'
cap = pyshark.FileCapture(pcap_path, display_filter='dns')

for i, packet in enumerate(cap):
    dns = packet.dns
    print(f"\n--- Packet {i} | {packet.sniff_time} ---")

    # Basic query/response info
    is_response = getattr(dns, 'flags_response', '0') == '1'
    print(f"Type: {'Response' if is_response else 'Query'}")

    if hasattr(dns, 'qry_name'):
        print(f"  Query name: {dns.qry_name}")
    if hasattr(dns, 'qry_type'):
        print(f"  Query type: {dns.qry_type}")

    # DNSSEC OK (DO) bit — signals resolver wants DNSSEC records
    if hasattr(dns, 'flags_authenticated'):
        print(f"  Flags: AD={getattr(dns, 'flags_authenticated', '?')} "
              f"CD={getattr(dns, 'flags_checkdisable', '?')}")

    # EDNS0 DO bit (in OPT record)
    if hasattr(dns, 'edns0_do'):
        print(f"  DO bit (DNSSEC requested): {dns.edns0_do}")

    # Response records — check for DNSSEC-specific RR types
    # Common type numbers: RRSIG=46, DNSKEY=48, DS=43, NSEC=47, NSEC3=50
    dnssec_types = {'46': 'RRSIG', '48': 'DNSKEY', '43': 'DS',
                     '47': 'NSEC', '50': 'NSEC3'}

    if hasattr(dns, 'resp_type'):
        resp_types = dns.resp_type.all_fields if hasattr(dns.resp_type, 'all_fields') else [dns.resp_type]
        for rt in resp_types:
            rt_val = str(rt.show if hasattr(rt, 'show') else rt)
            label = dnssec_types.get(rt_val, rt_val)
            print(f"  Response record type: {label}")

    # RRSIG-specific fields
    if hasattr(dns, 'rrsig_type_covered'):
        print(f"  RRSIG covers type: {dns.rrsig_type_covered}")
        print(f"  RRSIG algorithm: {getattr(dns, 'rrsig_algorithm', '?')}")
        print(f"  RRSIG signer name: {getattr(dns, 'rrsig_signers_name', '?')}")
        print(f"  RRSIG expiration: {getattr(dns, 'rrsig_expiration', '?')}")
        print(f"  RRSIG inception: {getattr(dns, 'rrsig_inception', '?')}")

    # DNSKEY-specific fields
    if hasattr(dns, 'dnskey_flags'):
        print(f"  DNSKEY flags: {dns.dnskey_flags}")
        print(f"  DNSKEY algorithm: {getattr(dns, 'dnskey_algorithm', '?')}")
        print(f"  DNSKEY key id: {getattr(dns, 'dnskey_key_id', '?')}")

    # DS-specific fields
    if hasattr(dns, 'ds_key_id'):
        print(f"  DS key tag: {dns.ds_key_id}")
        print(f"  DS digest type: {getattr(dns, 'ds_digest_type', '?')}")

    # NSEC/NSEC3 (proof of nonexistence)
    if hasattr(dns, 'nsec_next_domain_name'):
        print(f"  NSEC next domain: {dns.nsec_next_domain_name}")
    if hasattr(dns, 'nsec3_hash_algorithm'):
        print(f"  NSEC3 hash algorithm: {dns.nsec3_hash_algorithm}")

cap.close()


--- Packet 0 | 2013-01-23 02:48:50.954626 ---
Type: Query
  Query name: www.ietf.org
  Query type: 1
  Response record type: 41

--- Packet 1 | 2013-01-23 02:48:51.029064 ---
Type: Response
  Query name: www.ietf.org
  Query type: 1
  Flags: AD=1 CD=0
  Response record type: 1
  Response record type: RRSIG
  Response record type: 41
  RRSIG covers type: 1
  RRSIG algorithm: 5
  RRSIG signer name: ietf.org
  RRSIG expiration: ?
  RRSIG inception: ?

--- Packet 2 | 2013-01-23 02:48:51.029916 ---
Type: Query
  Query name: ietf.org
  Query type: 48
  Response record type: 41

--- Packet 3 | 2013-01-23 02:48:51.102520 ---
Type: Response
  Query name: ietf.org
  Query type: 48
  Flags: AD=1 CD=0
  Response record type: DNSKEY
  Response record type: DNSKEY
  Response record type: RRSIG
  Response record type: RRSIG
  Response record type: 41
  RRSIG covers type: 48
  RRSIG algorithm: 5
  RRSIG signer name: ietf.org
  RRSIG expiration: ?
  RRSIG inception: ?
  DNSKEY flags: 0x0101
  DNSKEY a